# 06 — SAR Resilience Scoring

> **CONTEXTUAL REMOTE-SENSING PRODUCT NOTICE**
> All SAR-derived resilience scores in this notebook are contextual remote-sensing
> products based on Sentinel-1 / PALSAR-2 backscatter data. They do **not**
> represent exact electrical asset geometry and **must not** be used for
> engineering or operational decisions without utility-authoritative verification.
> `sar_confidence` and topology `confidence` remain independent attributes
> throughout — they are never merged.

This notebook:
1. Loads the resilience risk index GeoJSON produced by `ExposureScorer`
2. Plots risk score distribution by LDC
3. Maps CRITICAL and HIGH substations with SAR flood-exposure context
4. Demonstrates spring-melt artefact suppression (months 3–4 per `sar_settings.yaml`)

Composite resilience score weights (from `sar_settings.yaml`):
```
sar_flood_exposure:   0.30
official_flood_zone:  0.25
elevation_inverse:    0.15
sar_change_tier:      0.15
saidi_percentile:     0.15
```
Risk labels: LOW (< 0.25) | MEDIUM (0.25–0.50) | HIGH (0.50–0.75) | CRITICAL (≥ 0.75)

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import date
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.cm as cm
from shapely.geometry import box, Point

from src.sar import check_sar_enabled, _SAR_DISCLAIMER
from src.sar.exposure_scorer import score_substation_resilience, _risk_label
from src.sar.flood_detector import _is_spring_melt_period
from src.ingestion.oeb_fetcher import OEBFetcher
from src.utils.config_loader import load_settings, load_sar_settings

cfg = load_settings()
sar_cfg = load_sar_settings().get("sar", {})

print("=" * 70)
print("SAR DISCLAIMER:")
print(_SAR_DISCLAIMER)
print("=" * 70)
print()

sar_enabled = check_sar_enabled()
print(f"SAR module enabled: {sar_enabled}")

bbox_cfg = cfg["region"]["bbox"]
BBOX = (bbox_cfg["west"], bbox_cfg["south"], bbox_cfg["east"], bbox_cfg["north"])

RESILIENCE_GEOJSON = Path(cfg["outputs"]["resilience_risk_geojson"])
print(f"Resilience GeoJSON: {RESILIENCE_GEOJSON}")
print(f"Exists: {RESILIENCE_GEOJSON.exists()}")

## 6.1  Load Resilience Risk Index

In [ ]:
# SAR DISCLAIMER: resilience scores are SAR_contextual_RS products
print(_SAR_DISCLAIMER)
print()

if RESILIENCE_GEOJSON.exists():
    risk_gdf = gpd.read_file(RESILIENCE_GEOJSON)
    print(f"Resilience index loaded: {len(risk_gdf):,} substations")
else:
    print(f"Resilience GeoJSON not found at {RESILIENCE_GEOJSON}")
    print("Generating synthetic resilience index for demo...")

    # Load substation nodes from GeoPackage and synthesise scores
    GPKG_PATH = Path(cfg["outputs"]["grid_gpkg"])
    nodes_gdf = gpd.read_file(GPKG_PATH, layer="nodes")
    substations = nodes_gdf[
        nodes_gdf["node_type"].isin(["tx_substation", "zone_substation", "dx_substation"])
    ].copy()

    # Add OEB LDC names via spatial join
    oeb = OEBFetcher()
    service_areas = oeb.fetch_service_areas()
    substations = substations.set_geometry("geometry")
    if substations.crs != service_areas.crs:
        substations = substations.to_crs(service_areas.crs)
    substations = gpd.sjoin(
        substations, service_areas[["ldc_name", "geometry"]],
        how="left", predicate="within"
    )
    substations = substations.rename(columns={"ldc_name": "ldc_name_area"})
    substations["ldc_name"] = substations["ldc_name_area"].fillna(substations.get("operator", ""))

    # Synthesise realistic risk score distribution using Beta distribution
    # Modelling: ~10% CRITICAL, ~25% HIGH, ~40% MEDIUM, ~25% LOW
    rng = np.random.default_rng(seed=99)
    n = len(substations)
    scores = np.concatenate([
        rng.beta(9, 2, int(n * 0.10)),    # CRITICAL cluster
        rng.beta(5, 3, int(n * 0.25)),    # HIGH cluster
        rng.beta(3, 5, int(n * 0.40)),    # MEDIUM cluster
        rng.beta(1.5, 8, n - int(n * 0.75)),  # LOW cluster
    ])
    rng.shuffle(scores)
    scores = scores[:n]

    substations["resilience_risk_score"] = scores.round(3)
    substations["risk_label"] = substations["resilience_risk_score"].apply(_risk_label)
    substations["sar_flood_exposure"] = rng.integers(0, 2, n)
    substations["sar_change_tier"] = rng.choice(
        ["none", "none", "moderate", "significant", "extreme"],
        size=n
    )
    substations["in_official_floodzone"] = rng.choice([True, False], size=n, p=[0.15, 0.85])
    substations["elevation_m"] = rng.uniform(75, 220, n).round(1)
    substations["saidi_percentile"] = rng.uniform(0.1, 0.9, n).round(3)
    substations["source"] = "SAR_contextual_RS"
    substations["is_exact_asset_geometry"] = False
    substations["sar_confidence"] = substations["sar_flood_exposure"].apply(
        lambda x: "unvalidated" if x else None
    )
    substations["disclaimer_text"] = _SAR_DISCLAIMER

    risk_gdf = substations
    print(f"Synthetic resilience index: {len(risk_gdf):,} substations")

print()
print("Risk label distribution:")
print(risk_gdf["risk_label"].value_counts().to_string())
print()
print("Source / confidence check:")
print(f"  source: {risk_gdf['source'].unique()}")
print(f"  is_exact_asset_geometry: {risk_gdf['is_exact_asset_geometry'].unique()}")
print(f"  sar_confidence values: {risk_gdf['sar_confidence'].value_counts(dropna=False).to_dict()}")

## 6.2  Risk Score Distribution by LDC

In [ ]:
# SAR DISCLAIMER: risk scores derived from SAR_contextual_RS products
print(_SAR_DISCLAIMER)
print()

# Aggregate risk stats by LDC
ldc_col = "ldc_name" if "ldc_name" in risk_gdf.columns else "operator"

ldc_stats = (
    risk_gdf
    .groupby(ldc_col)["resilience_risk_score"]
    .agg(["mean", "max", "count"])
    .round(3)
    .rename(columns={"mean": "mean_score", "max": "max_score", "count": "substations"})
    .sort_values("mean_score", ascending=False)
    .reset_index()
)

# Count risk tiers per LDC
for label in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]:
    ldc_tier = (
        risk_gdf[risk_gdf["risk_label"] == label]
        .groupby(ldc_col)
        .size()
        .rename(label)
    )
    ldc_stats = ldc_stats.merge(ldc_tier, on=ldc_col, how="left").fillna({label: 0})
    ldc_stats[label] = ldc_stats[label].astype(int)

print(f"LDC resilience summary ({len(ldc_stats)} LDCs):")
print(ldc_stats.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle(
    "Resilience Risk Score by LDC\n"
    "[source=SAR_contextual_RS | is_exact_asset_geometry=False]",
    fontsize=12, fontweight="bold"
)

risk_colors = {"CRITICAL": "#c0392b", "HIGH": "#e67e22", "MEDIUM": "#f1c40f", "LOW": "#27ae60"}

# Stacked bar: count of risk tiers per LDC
ax = axes[0]
top_ldcs = ldc_stats.head(15)
y = range(len(top_ldcs))
left = np.zeros(len(top_ldcs))
for label in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]:
    if label in top_ldcs.columns:
        vals = top_ldcs[label].values
        bars = ax.barh(list(y), vals, left=left, color=risk_colors[label],
                       label=label, edgecolor="white", linewidth=0.5)
        left += vals

ax.set_yticks(list(y))
ax.set_yticklabels(top_ldcs[ldc_col].values, fontsize=8)
ax.set_xlabel("Substation count")
ax.set_title("Risk Tier Distribution (top 15 LDCs)")
ax.legend(fontsize=9, loc="lower right")

# Box plot: risk score distribution per LDC
ax = axes[1]
top_ldc_names = top_ldcs[ldc_col].tolist()
box_data = [
    risk_gdf[risk_gdf[ldc_col] == ldc]["resilience_risk_score"].dropna().tolist()
    for ldc in top_ldc_names
]
box_data = [d for d in box_data if d]  # drop empty
ldc_labels_nonzero = [top_ldc_names[i] for i, d in enumerate([
    risk_gdf[risk_gdf[ldc_col] == ldc]["resilience_risk_score"].dropna().tolist()
    for ldc in top_ldc_names
]) if d]

if box_data:
    bp = ax.boxplot(box_data, vert=False, patch_artist=True, notch=False,
                    medianprops=dict(color="black", linewidth=1.5))
    cmap_colors = cm.RdYlGn_r(np.linspace(0.2, 0.9, len(box_data)))
    for patch, color in zip(bp["boxes"], cmap_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_yticks(range(1, len(ldc_labels_nonzero) + 1))
    ax.set_yticklabels(ldc_labels_nonzero, fontsize=8)
    ax.axvline(0.75, color="#c0392b", linestyle="--", linewidth=1.2, label="CRITICAL threshold")
    ax.axvline(0.50, color="#e67e22", linestyle="--", linewidth=1.2, label="HIGH threshold")
    ax.axvline(0.25, color="#f1c40f", linestyle="--", linewidth=1.2, label="MEDIUM threshold")
    ax.legend(fontsize=8, loc="lower right")
    ax.set_xlabel("Resilience risk score (0=low risk, 1=high risk)")
    ax.set_title("Score Distribution per LDC")
    ax.set_xlim(0, 1)

plt.tight_layout()
plt.savefig("../data/outputs/06_risk_score_by_ldc.png", dpi=150, bbox_inches="tight")
plt.show()

## 6.3  CRITICAL and HIGH Substation Map with SAR Flood Context

In [ ]:
# SAR DISCLAIMER: flood exposure context is a SAR_contextual_RS product
print(_SAR_DISCLAIMER)
print()

# Filter to CRITICAL and HIGH substations
high_risk = risk_gdf[
    risk_gdf["risk_label"].isin(["CRITICAL", "HIGH"])
].copy()

print(f"CRITICAL substations: {(high_risk['risk_label'] == 'CRITICAL').sum():,}")
print(f"HIGH substations:     {(high_risk['risk_label'] == 'HIGH').sum():,}")
print()
print("sar_confidence check (high-risk only):")
print(high_risk["sar_confidence"].value_counts(dropna=False).to_string())
print()

n_unvalidated = (high_risk["sar_confidence"] == "unvalidated").sum()
n_uncertain   = (high_risk["sar_confidence"] == "uncertain").sum()
if n_uncertain > 0:
    print(f"WARNING: {n_uncertain} high-risk substations have sar_confidence='uncertain'")
    print("         (spring-melt artefact — see Section 6.4 for suppression demo)")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
ax.set_facecolor("#1a252f")

# Background: all substations
all_sub = risk_gdf[risk_gdf.geometry.geom_type == "Point"]
low_med = all_sub[all_sub["risk_label"].isin(["LOW", "MEDIUM"])]
if len(low_med) > 0:
    low_med.plot(ax=ax, color="#5d6d7e", markersize=15, alpha=0.35, zorder=2)

# Plot CRITICAL and HIGH substations
for label, marker, size, color in [
    ("CRITICAL", "*",  200, "#e74c3c"),
    ("HIGH",     "^",  120, "#e67e22"),
]:
    subset = high_risk[high_risk["risk_label"] == label]
    pt_subset = subset[subset.geometry.geom_type == "Point"]
    if len(pt_subset) > 0:
        pt_subset.plot(ax=ax, color=color, markersize=size, marker=marker,
                       zorder=5, edgecolors="white", linewidth=0.8,
                       label=f"{label} ({len(pt_subset):,})")

# Highlight SAR-flood-exposed CRITICAL substations with a ring
if "sar_flood_exposure" in high_risk.columns:
    flood_exposed = high_risk[
        (high_risk["risk_label"] == "CRITICAL") &
        (high_risk["sar_flood_exposure"] == 1) &
        (high_risk.geometry.geom_type == "Point")
    ]
    if len(flood_exposed) > 0:
        flood_exposed.plot(ax=ax, facecolor="none", edgecolors="#3498db",
                           markersize=280, marker="o", linewidth=2,
                           zorder=4, label=f"+ SAR flood exposure ({len(flood_exposed)})")

ax.set_xlim(BBOX[0], BBOX[2])
ax.set_ylim(BBOX[1], BBOX[3])
ax.set_xlabel("Longitude", color="white")
ax.set_ylabel("Latitude", color="white")
ax.tick_params(colors="white")
ax.set_title(
    "CRITICAL and HIGH Substations — SAR Flood Exposure Context\n"
    "source=SAR_contextual_RS | is_exact_asset_geometry=False\n"
    "Blue ring = SAR water-extent mask overlap (sar_confidence='unvalidated')",
    fontsize=10, fontweight="bold", color="white"
)
ax.legend(fontsize=10, loc="lower right", facecolor="#2c3e50", labelcolor="white")

# SAR disclaimer as text box on figure
fig.text(
    0.01, 0.01,
    f"NOTE: {_SAR_DISCLAIMER[:120]}...",
    fontsize=6, color="#aab7b8", wrap=True,
    transform=fig.transFigure
)

plt.tight_layout()
plt.savefig("../data/outputs/06_high_risk_substation_map.png", dpi=150, bbox_inches="tight")
plt.show()

## 6.4  Spring Melt Artefact Suppression Demo

In [ ]:
# SAR DISCLAIMER: flood detection subject to spring-melt artefact
print(_SAR_DISCLAIMER)
print()

spring_melt_months = sar_cfg.get("artefact_suppression", {}).get("spring_melt_months", [3, 4])
flag_spring_melt   = sar_cfg.get("artefact_suppression", {}).get("flag_spring_melt", True)
min_scenes         = sar_cfg.get("artefact_suppression", {}).get("min_scenes_for_confirmation", 2)

print(f"Spring melt artefact suppression config:")
print(f"  flag_spring_melt:             {flag_spring_melt}")
print(f"  spring_melt_months:           {spring_melt_months}  (March–April)")
print(f"  min_scenes_for_confirmation:  {min_scenes}")
print()

# Demonstrate suppression logic on example event dates
test_dates = [
    "2024-02-15",  # February — not spring melt
    "2024-03-01",  # March — spring melt window
    "2024-03-28",  # March — spring melt window
    "2024-04-18",  # April — spring melt window
    "2024-05-10",  # May — outside window
    "2024-07-22",  # July — outside window
    "2025-03-15",  # March 2025 — spring melt window
]

print(f"{'Event date':<14}  {'Month':<6}  {'Spring melt?':<14}  SAR confidence impact")
print("-" * 65)
for d in test_dates:
    month = date.fromisoformat(d).month
    is_melt = _is_spring_melt_period(d, sar_cfg)
    conf_impact = "sar_confidence='uncertain'" if is_melt else "sar_confidence='unvalidated' (normal)"
    flag = "YES" if is_melt else "no"
    print(f"  {d:<14}  {month:<6}  {flag:<14}  {conf_impact}")

In [ ]:
# Visualise the effect of spring-melt suppression on flood-exposed substations
# Scenario: compare scores with vs without suppression for an April 18 2024 event

EVENT_DATE = "2024-04-18"
is_melt_event = _is_spring_melt_period(EVENT_DATE, sar_cfg)

print(f"Event date: {EVENT_DATE}")
print(f"Spring melt window: {is_melt_event}")
print()

# Simulate two scoring runs: one raw, one with suppression applied
rng = np.random.default_rng(seed=11)
n_demo = 80
raw_scores = rng.beta(4, 3, n_demo)  # biased toward moderate risk

# Suppression: reduce sar_flood_exposure weight for uncertain scenes
# (simulated: ~30% of detections would be artefacts in spring-melt window)
suppressed_scores = raw_scores.copy()
if is_melt_event:
    # Identify "uncertain" detections (simulated)
    uncertain_mask = rng.random(n_demo) < 0.30
    # Suppress: reduce score by 0.30 * sar_flood_exposure_weight for uncertain pixels
    suppressed_scores[uncertain_mask] = np.clip(
        suppressed_scores[uncertain_mask] - 0.30 * 0.30, 0, 1
    )

raw_labels      = [_risk_label(s) for s in raw_scores]
suppressed_labels = [_risk_label(s) for s in suppressed_scores]

print("Risk label distribution — RAW (no suppression):")
print(pd.Series(raw_labels).value_counts().to_string())
print()
print(f"Risk label distribution — SUPPRESSED (spring-melt correction, {EVENT_DATE}):")
print(pd.Series(suppressed_labels).value_counts().to_string())
print()
downgraded = (pd.Series(raw_labels) != pd.Series(suppressed_labels)).sum()
print(f"Substations downgraded after suppression: {downgraded} / {n_demo}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    f"Spring Melt Artefact Suppression Demo — Event date: {EVENT_DATE}\n"
    f"[source=SAR_contextual_RS | is_exact_asset_geometry=False | months={spring_melt_months}]",
    fontsize=11, fontweight="bold"
)

score_bins = np.linspace(0, 1, 26)

# Raw scores
ax = axes[0]
ax.hist(raw_scores, bins=score_bins, color="#e74c3c", edgecolor="white",
        alpha=0.80, label="Raw scores")
for threshold, color, label in [
    (0.75, "#c0392b", "CRITICAL"),
    (0.50, "#e67e22", "HIGH"),
    (0.25, "#f1c40f", "MEDIUM"),
]:
    ax.axvline(threshold, color=color, linestyle="--", linewidth=1.5, label=label)
ax.set_xlabel("Resilience risk score")
ax.set_ylabel("Substation count")
ax.set_title(f"RAW — {EVENT_DATE}\nsar_confidence='unvalidated' (no suppression)")
ax.legend(fontsize=8)
ax.set_xlim(0, 1)

# Suppressed scores
ax = axes[1]
ax.hist(suppressed_scores, bins=score_bins, color="#27ae60", edgecolor="white",
        alpha=0.80, label="Suppressed scores")
for threshold, color, label in [
    (0.75, "#c0392b", "CRITICAL"),
    (0.50, "#e67e22", "HIGH"),
    (0.25, "#f1c40f", "MEDIUM"),
]:
    ax.axvline(threshold, color=color, linestyle="--", linewidth=1.5, label=label)
ax.set_xlabel("Resilience risk score")
ax.set_title(
    f"SUPPRESSED — {EVENT_DATE}\n"
    f"sar_confidence='uncertain' applied\n"
    f"{downgraded} substations downgraded"
)
ax.legend(fontsize=8)
ax.set_xlim(0, 1)

plt.tight_layout()
plt.savefig("../data/outputs/06_spring_melt_suppression_demo.png", dpi=150, bbox_inches="tight")
plt.show()

print()
print("Key takeaway: spring-melt artefact suppression reduces over-detection of")
print("flood exposure in March–April scenes by downgrading sar_confidence to")
print("'uncertain' and reducing the sar_flood_exposure weight contribution.")
print("Final scores must still be validated against ECCC / CEMS flood zone data.")

## 6.5  Resilience Score Component Breakdown

In [ ]:
# SAR DISCLAIMER: component scores are SAR_contextual_RS products
print(_SAR_DISCLAIMER)
print()

weights = sar_cfg.get("resilience_score_weights", {
    "sar_flood_exposure":  0.30,
    "official_flood_zone": 0.25,
    "elevation_inverse":   0.15,
    "sar_change_tier":     0.15,
    "saidi_percentile":    0.15,
})

critical_subs = risk_gdf[risk_gdf["risk_label"] == "CRITICAL"].head(10)

print(f"Top CRITICAL substations (score >= 0.75):")
print(f"{'Name':<35} {'Score':>7} {'SAR Flood':>10} {'Flood Zone':>11} {'Change Tier':>12} {'SAIDI pct':>10}")
print("-" * 90)
for _, row in critical_subs.iterrows():
    name = str(row.get("name", row.get("node_id", "Unknown")))[:34]
    score    = float(row.get("resilience_risk_score", 0))
    flood    = int(row.get("sar_flood_exposure", 0))
    in_zone  = bool(row.get("in_official_floodzone", False))
    chg_tier = str(row.get("sar_change_tier", "none"))
    saidi    = float(row.get("saidi_percentile", 0.5))
    print(f"  {name:<33} {score:>7.3f} {flood:>10} {str(in_zone):>11} {chg_tier:>12} {saidi:>10.3f}")

print()
print("Score weight breakdown:")
for component, w in weights.items():
    print(f"  {component:<28} {w:.2f}")

## 6.6  Summary

In [ ]:
# SAR DISCLAIMER — final required print
print(_SAR_DISCLAIMER)
print()

n_critical = (risk_gdf["risk_label"] == "CRITICAL").sum()
n_high     = (risk_gdf["risk_label"] == "HIGH").sum()
n_medium   = (risk_gdf["risk_label"] == "MEDIUM").sum()
n_low      = (risk_gdf["risk_label"] == "LOW").sum()
n_sar_flood = int(risk_gdf["sar_flood_exposure"].fillna(0).astype(int).sum()) \
    if "sar_flood_exposure" in risk_gdf.columns else 0
n_eccc_zone = int(risk_gdf["in_official_floodzone"].fillna(False).sum()) \
    if "in_official_floodzone" in risk_gdf.columns else 0

print("=" * 68)
print("  SAR RESILIENCE SCORING — SUMMARY")
print("=" * 68)
print(f"  source:                    SAR_contextual_RS")
print(f"  is_exact_asset_geometry:   False")
print(f"  Total substations scored:  {len(risk_gdf):,}")
print()
print(f"  Risk label breakdown:")
print(f"    CRITICAL (score >= 0.75):  {n_critical:>5,}")
print(f"    HIGH     (0.50–0.75):      {n_high:>5,}")
print(f"    MEDIUM   (0.25–0.50):      {n_medium:>5,}")
print(f"    LOW      (< 0.25):         {n_low:>5,}")
print()
print(f"  SAR flood exposure flag:    {n_sar_flood:,} substations")
print(f"  ECCC official flood zone:   {n_eccc_zone:,} substations")
print()
print(f"  Spring melt artefact:")
print(f"    Flagged months:           {spring_melt_months}")
print(f"    Suppression active:       {flag_spring_melt}")
print(f"    Min confirmation scenes:  {min_scenes}")
print()
print("  Score weights:")
for k, v in weights.items():
    print(f"    {k:<28} {v:.2f}")
print("=" * 68)